# Khmer Grammar Checking — Improved Binary Classification

ITM 454 NLP course project. Two model paths:

1. **BiGRU + attention + feature fusion** (`train.py`): FastText word vectors
   concatenated with per-token POS embeddings, plus 5 scalar features.
   Weighted loss + tuned decision threshold to counter the false-positive bias.
2. **Khmer transformer** (`train_transformer.py`): fine-tunes
   `seanghay/xlm-roberta-khmer-small` directly on raw text.

The old dataset had a **"space" leak**: 100% of Wrong rows contained a space
(space-separated scrambled tokens) vs 49.9% of Right rows. A model could cheat
on spacing alone. See `scripts/generate_errors.py` for the fixed data generator
(negatives joined without spaces) and run the data-quality cell below to verify.
Full change log: `CHANGES.md`. Backup of the pre-change project: `../KGCBC`.


## 1. Setup

Install dependencies and download the Khmer FastText model if missing.
In Colab, make sure `train_data.csv` and this notebook are in the working
directory (upload them, or mount Drive and `%cd` into the folder).


In [ ]:
# !pip install -r requirements.txt
# !wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.km.300.bin.gz
# !gunzip cc.km.300.bin.gz


## 2. Write the project modules

Each source file is written to the runtime filesystem verbatim so the
scripts can run unchanged.


In [ ]:
!mkdir -p src/data src/features src/models src/pipeline scripts

In [ ]:
%writefile src/__init__.py


In [ ]:
%writefile src/config.py
import torch

SEED = 42
FASTTEXT_MODEL_PATH = "cc.km.300.bin"
DATA_PATH = "train_data.csv"
MODEL_SAVE_PATH = "gru_model.pth"

EMBEDDING_DIM = 300
HIDDEN_DIM = 256
OUTPUT_DIM = 2
N_LAYERS = 2
DROPOUT = 0.5
LEARNING_RATE = 0.001
N_EPOCHS = 15
BATCH_SIZE = 32
PATIENCE = 3
CLIP_VALUE = 1.0

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

FEATURE_COLUMNS = [
    "oov_ratio",
    "dep_grammar_score",
    "sentence_length",
    "pos_diversity",
    "avg_word_length",
]

USE_FEATURE_FUSION = True
NUM_EXTRA_FEATURES = len(FEATURE_COLUMNS)

# Heavier weight on Wrong (0) reduces over-predicting "Right" (the known false-positive bias).
CLASS_WEIGHTS = [1.3, 1.0]

# Per-token POS embedding (sequence grammar signal), fed into the GRU alongside word vectors.
POS_TAGS = [
    "AB", "AUX", "CC", "CD", "DBL", "DT", "ETC", "IN", "JJ", "KAN",
    "M", "NN", "PA", "PN", "PRO", "QT", "RB", "RPN", "SYM", "UH",
    "VB", "VB_JJ", "VCOM", "UNKNOWN",
]
POS_EMBEDDING_DIM = 32


In [ ]:
%writefile src/utils.py
import os
import ast
import numpy as np
import torch
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from src.config import FEATURE_COLUMNS, BATCH_SIZE, DEVICE
from src.data.dataset import KhmerTextDataset, collate_batch


def load_fasttext_model(model_path):
    if not os.path.exists(model_path):
        print(f"FastText model not found at {model_path}, downloading...")
        import fasttext.util
        fasttext.util.download_model('km', if_exists='ignore')
        downloaded_path = f"cc.km.300.bin"
        if os.path.exists(downloaded_path):
            model_path = downloaded_path
        else:
            raise FileNotFoundError(
                f"Downloaded model not found at {downloaded_path}. "
                f"Please download cc.km.300.bin manually from https://fasttext.cc/docs/en/crawl-vectors.html"
            )
    import fasttext
    return fasttext.load_model(model_path)


def load_and_split_data(df_path="train_data.csv", feature_columns=None):
    if feature_columns is None:
        feature_columns = FEATURE_COLUMNS
    df = pd.read_csv(df_path, encoding="utf-8-sig")
    if "tokens" in df.columns:
        df["tokens"] = df["tokens"].apply(ast.literal_eval)
    if "pos_tags" in df.columns:
        df["pos_tags"] = df["pos_tags"].apply(ast.literal_eval)
    available_features = [col for col in feature_columns if col in df.columns]
    X = df[available_features]
    y = df["sentence_correct"]
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    print(f"Train set: {len(X_train)} samples")
    print(f"Validation set: {len(X_val)} samples")
    print(f"Test set: {len(X_test)} samples")
    return df, X_train, X_val, X_test, y_train, y_val, y_test, scaler, X_train_scaled, X_val_scaled, X_test_scaled


def build_feature_tensor(df, indices, feature_columns, scaler=None):
    features = df.loc[indices, feature_columns].values
    if scaler is not None:
        features = scaler.transform(features)
    return torch.FloatTensor(features)


def create_dataloaders(train_dataset, val_dataset, test_dataset, batch_size=BATCH_SIZE):
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch
    )
    return train_loader, val_loader, test_loader


def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)


In [ ]:
%writefile src/data/__init__.py
from .dataset import KhmerTextDataset, collate_batch
from .pos_tagger import KhmerPOSTagger

__all__ = ["KhmerTextDataset", "collate_batch", "KhmerPOSTagger"]


In [ ]:
%writefile src/data/pos_tagger.py
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline


class KhmerPOSTagger:
    def __init__(self, model_name="seanghay/khmer-pos-roberta"):
        print(f"Loading POS tagger: {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.pipeline = pipeline(
            "token-classification",
            model=self.model,
            tokenizer=self.tokenizer,
            aggregation_strategy="simple",
        )
        print("POS tagger loaded successfully")

    def tag_sentence(self, sentence):
        try:
            tags = self.pipeline(sentence)
            tokens = [t["word"] for t in tags]
            pos_tags = [t["entity_group"] for t in tags]
            return tokens, pos_tags
        except Exception as e:
            print(f"POS tagging error for sentence: {sentence[:50]}... Error: {e}")
            return [], []

    def tag_dataframe(self, df, text_column="text"):
        print(f"Tagging {len(df)} sentences...")
        tokens_list = []
        pos_tags_list = []
        for sentence in tqdm(df[text_column], desc="POS Tagging"):
            tokens, pos_tags = self.tag_sentence(sentence)
            tokens_list.append(tokens)
            pos_tags_list.append(pos_tags)
        df["tokens"] = tokens_list
        df["pos_tags"] = pos_tags_list
        print("POS tagging complete")
        print("Added 'tokens' and 'pos_tags' columns")
        return df


In [ ]:
%writefile src/data/dataset.py
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

from src.config import POS_TAGS


POS_TO_ID = {tag: i for i, tag in enumerate(POS_TAGS)}
POS_UNKNOWN = POS_TO_ID["UNKNOWN"]


def pos_tags_to_ids(pos_tags):
    return [POS_TO_ID.get(tag, POS_UNKNOWN) for tag in pos_tags]


class KhmerTextDataset(Dataset):
    _embedding_cache = {}

    def __init__(self, tokens_list, pos_tags_list, labels, embedding_model, use_cache=True):
        self.tokens_list = tokens_list
        self.pos_tags_list = pos_tags_list
        self.labels = labels
        self.embedding_model = embedding_model
        self.embedding_dim = embedding_model.get_dimension()
        self.use_cache = use_cache
        if self.use_cache:
            self._build_cache()

    def _build_cache(self):
        unique_tokens = set()
        for tokens in self.tokens_list:
            unique_tokens.update(tokens)
        new_tokens = unique_tokens - set(self._embedding_cache.keys())
        if new_tokens:
            print(f"Caching embeddings for {len(new_tokens)} new unique tokens...")
            for token in new_tokens:
                try:
                    vec = self.embedding_model.get_word_vector(token)
                except Exception:
                    vec = np.zeros(self.embedding_dim)
                self._embedding_cache[token] = vec
            print(f"Total cached tokens: {len(self._embedding_cache)}")

    def __len__(self):
        return len(self.tokens_list)

    def __getitem__(self, idx):
        tokens = self.tokens_list[idx]
        pos_tags = self.pos_tags_list[idx]
        label = self.labels[idx]
        embeddings = []
        for token in tokens:
            if self.use_cache and token in self._embedding_cache:
                vec = self._embedding_cache[token]
            else:
                try:
                    vec = self.embedding_model.get_word_vector(token)
                except Exception:
                    vec = np.zeros(self.embedding_dim)
            embeddings.append(vec)
        embeddings = torch.FloatTensor(np.array(embeddings))
        pos_ids = torch.LongTensor(pos_tags_to_ids(pos_tags))
        label = torch.LongTensor([label])
        return embeddings, pos_ids, label, len(tokens)

    @classmethod
    def clear_cache(cls):
        cls._embedding_cache.clear()
        print("Embedding cache cleared")


def collate_batch(batch):
    embeddings_list, pos_ids_list, labels_list, lengths_list = zip(*batch)
    padded_embeddings = pad_sequence(embeddings_list, batch_first=True)
    padded_pos_ids = pad_sequence(pos_ids_list, batch_first=True, padding_value=POS_UNKNOWN)
    labels = torch.cat(labels_list)
    lengths = torch.LongTensor(lengths_list)
    return padded_embeddings, padded_pos_ids, labels, lengths


In [ ]:
%writefile src/features/__init__.py
from .oov import EmbeddingOOVCalculator
from .grammar import SimplePOSGrammarExtractor
from .structural import StructuralFeatureExtractor

__all__ = [
    "EmbeddingOOVCalculator",
    "SimplePOSGrammarExtractor",
    "StructuralFeatureExtractor",
]


In [ ]:
%writefile src/features/oov.py
import numpy as np
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm


class EmbeddingOOVCalculator:
    def __init__(self, embedding_model=None):
        self.embedding_model = embedding_model
        self.vocab = set()
        if embedding_model:
            self._load_vocab_from_model()

    def _load_vocab_from_model(self):
        try:
            if hasattr(self.embedding_model, "get_words"):
                self.vocab = set(self.embedding_model.get_words())
            else:
                raise AttributeError("Model doesn't have recognized vocabulary interface")
            print(f"Loaded {len(self.vocab):,} words from embedding model")
        except Exception as e:
            print(f"Error loading vocabulary from model: {e}")

    def is_in_vocabulary(self, word):
        return word.strip() in self.vocab

    def calculate_oov_ratio(self, tokens):
        if isinstance(tokens, str):
            words = tokens.split()
        else:
            words = tokens
        if not words:
            return 0.0
        oov_count = sum(1 for word in words if not self.is_in_vocabulary(word))
        return oov_count / len(words)

    def calculate_oov_details(self, tokens):
        if isinstance(tokens, str):
            words = tokens.split()
        else:
            words = tokens
        oov_words = []
        known_words = []
        for word in words:
            if self.is_in_vocabulary(word):
                known_words.append(word)
            else:
                oov_words.append(word)
        return {
            "total_words": len(words),
            "known_words": len(known_words),
            "oov_words": len(oov_words),
            "oov_ratio": len(oov_words) / max(len(words), 1),
            "oov_word_list": oov_words,
            "known_word_list": known_words,
            "vocabulary_coverage": len(known_words) / max(len(words), 1),
        }

    def add_oov_features(self, df, text_col="tokens"):
        df = df.copy()
        print(f"Calculating OOV features for {len(df):,} texts...")
        oov_ratios = []
        vocab_coverages = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc="OOV Analysis"):
            text = row[text_col]
            details = self.calculate_oov_details(text)
            oov_ratios.append(details["oov_ratio"])
            vocab_coverages.append(details["vocabulary_coverage"])
        df["oov_ratio"] = oov_ratios
        df["vocab_coverage"] = vocab_coverages
        print(f"Mean OOV ratio: {df['oov_ratio'].mean():.3f}")
        print(f"Mean vocab coverage: {df['vocab_coverage'].mean():.3f}")
        return df

    def extract_unknown_words(self, df, text_col="tokens", output_file="unknown_words.csv"):
        print("Extracting unknown words...")
        unknown_words = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting Unknown Words"):
            text = row[text_col]
            details = self.calculate_oov_details(text)
            unknown_words.extend(details["oov_word_list"])
        word_counts = Counter(unknown_words)
        unknown_df = pd.DataFrame(
            [{"word": word, "frequency": count} for word, count in word_counts.most_common()]
        )
        print(f"Found {len(unknown_df):,} unique unknown words")
        if output_file and len(unknown_df) > 0:
            unknown_df.to_csv(output_file, index=False, encoding="utf-8")
            print(f"Saved to {output_file}")
        return unknown_df

    def evaluate_text_quality(self, text):
        details = self.calculate_oov_details(text)
        vocab_score = details["vocabulary_coverage"] * 100
        if vocab_score >= 95:
            quality = "Excellent"
            message = "Vocabulary is excellent"
        elif vocab_score >= 85:
            quality = "Good"
            message = "Good vocabulary usage"
        elif vocab_score >= 70:
            quality = "Fair"
            message = "Some vocabulary issues detected"
        else:
            quality = "Poor"
            message = "Many unknown words detected"
        return {
            "quality": quality,
            "vocab_score": vocab_score,
            "message": message,
            "total_words": details["total_words"],
            "known_words": details["known_words"],
            "unknown_words": details["oov_words"],
            "unknown_word_list": details["oov_word_list"],
        }


In [ ]:
%writefile src/features/grammar.py
import pandas as pd


class SimplePOSGrammarExtractor:
    @staticmethod
    def calculate_grammar_score(pos_tags):
        if not pos_tags or len(pos_tags) == 0:
            return 0.0

        score = 0.0

        has_noun = any(t.startswith("NN") for t in pos_tags)
        has_pronoun = any(t.startswith("PR") for t in pos_tags)
        has_verb = any(t.startswith("VB") for t in pos_tags)
        has_aux = any(t == "AUX" for t in pos_tags)
        has_adj = any(t.startswith("JJ") for t in pos_tags)
        has_adv = any(t.startswith("RB") for t in pos_tags)
        has_conj = any(t == "CC" for t in pos_tags)
        has_det = any(t.startswith("DT") for t in pos_tags)
        has_num = any(t.startswith("CD") for t in pos_tags)
        has_adp = any(t.startswith("IN") for t in pos_tags)

        has_subject = has_noun or has_pronoun
        has_predicate = has_verb or has_aux

        if has_subject and has_predicate:
            score += 0.45
        elif has_predicate:
            score += 0.15
        elif has_subject and not has_predicate:
            score -= 0.1

        if has_aux:
            score += 0.1
        if has_adj:
            score += 0.05
        if has_adv:
            score += 0.05
        if has_conj:
            score += 0.1
        if has_det or has_num or has_adp:
            score += 0.05

        unique_pos_count = len(set(pos_tags))
        score += min(0.1, unique_pos_count * 0.025)

        if not has_verb and not has_aux:
            score -= 0.25

        if len(pos_tags) <= 2:
            score -= 0.2

        if has_noun and not has_verb and not has_aux and not has_adj:
            score -= 0.2

        return max(0.0, min(1.0, score))

    @staticmethod
    def has_complete_clause(pos_tags):
        if not pos_tags or len(pos_tags) == 0:
            return 0
        has_noun = any(tag.startswith("NN") or tag.startswith("PR") for tag in pos_tags)
        has_verb = any(tag.startswith("VB") or tag == "AUX" for tag in pos_tags)
        if has_noun and has_verb:
            return 1
        return 0

    def extract_features(self, df):
        print("Extracting improved POS-based grammar features...")
        df["dep_grammar_score"] = df["pos_tags"].apply(self.calculate_grammar_score)
        print(f"Mean grammar score: {df['dep_grammar_score'].mean():.3f}")
        print(f"Grammar score std:  {df['dep_grammar_score'].std():.3f}")
        return df


In [ ]:
%writefile src/features/structural.py
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


class StructuralFeatureExtractor:
    @staticmethod
    def sentence_length(tokens):
        if isinstance(tokens, list):
            return len(tokens)
        return 0

    @staticmethod
    def pos_diversity(pos_tags):
        if not pos_tags or len(pos_tags) == 0:
            return 0.0
        return len(set(pos_tags)) / len(pos_tags)

    @staticmethod
    def avg_word_length(tokens):
        if not tokens or len(tokens) == 0:
            return 0.0
        lengths = [len(str(w)) for w in tokens]
        return np.mean(lengths)

    def extract_features(self, df):
        print("Extracting structural features...")
        df["sentence_length"] = df["tokens"].apply(self.sentence_length)
        df["pos_diversity"] = df["pos_tags"].apply(self.pos_diversity)
        df["avg_word_length"] = df["tokens"].apply(self.avg_word_length)
        print(f"Mean sentence length: {df['sentence_length'].mean():.2f}")
        print(f"Mean POS diversity:   {df['pos_diversity'].mean():.3f}")
        print(f"Mean word length:     {df['avg_word_length'].mean():.2f}")
        return df


In [ ]:
%writefile src/models/__init__.py
from .gru import GRUClassifier, ImprovedGRUClassifier, AttentionPooling
from .train_utils import EarlyStopping, train_gru_epoch, evaluate_gru
from .train_utils import train_gru_epoch_with_features, evaluate_gru_with_features

__all__ = [
    "GRUClassifier",
    "ImprovedGRUClassifier",
    "AttentionPooling",
    "EarlyStopping",
    "train_gru_epoch",
    "evaluate_gru",
    "train_gru_epoch_with_features",
    "evaluate_gru_with_features",
]


In [ ]:
%writefile src/models/gru.py
import torch
import torch.nn as nn


class GRUClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, dropout=0.5):
        super(GRUClassifier, self).__init__()
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=True,
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, embedded_text, text_lengths):
        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded_text, text_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_output, hidden = self.gru(packed_embedded)
        hidden = self.dropout(torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1))
        output = self.fc(hidden)
        return output


class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super(AttentionPooling, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, sequence_output, lengths):
        scores = self.attention(sequence_output).squeeze(-1)
        batch_size, max_len = sequence_output.size(0), sequence_output.size(1)
        mask = (
            torch.arange(max_len, device=sequence_output.device).unsqueeze(0)
            < lengths.unsqueeze(1)
        )
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), sequence_output).squeeze(1)
        return context, weights


class ImprovedGRUClassifier(nn.Module):
    def __init__(
        self,
        embedding_dim,
        hidden_dim,
        output_dim,
        n_layers,
        dropout=0.5,
        num_extra_features=0,
        use_feature_fusion=False,
        pos_vocab_size=0,
        pos_embedding_dim=0,
    ):
        super(ImprovedGRUClassifier, self).__init__()
        self.use_feature_fusion = use_feature_fusion
        self.num_extra_features = num_extra_features
        self.use_pos = pos_vocab_size > 0 and pos_embedding_dim > 0
        self.pos_embedding = (
            nn.Embedding(pos_vocab_size, pos_embedding_dim) if self.use_pos else None
        )
        self.gru = nn.GRU(
            embedding_dim + (pos_embedding_dim if self.use_pos else 0),
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=True,
        )
        self.attention = AttentionPooling(hidden_dim * 2)
        pooled_dim = hidden_dim * 2 * 2
        self.layer_norm = nn.LayerNorm(pooled_dim)
        mlp_input_dim = pooled_dim
        if use_feature_fusion and num_extra_features > 0:
            mlp_input_dim += num_extra_features
        self.classifier = nn.Sequential(
            nn.Linear(mlp_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, embedded_text, text_lengths, extra_features=None, pos_ids=None):
        if self.use_pos:
            pos_emb = self.pos_embedding(pos_ids)
            embedded_text = torch.cat([embedded_text, pos_emb], dim=-1)
        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded_text, text_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_output, hidden = self.gru(packed_embedded)
        sequence_output, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)
        attention_context, attention_weights = self.attention(sequence_output, text_lengths)
        batch_size, max_len, hidden_size = sequence_output.size()
        mask = (
            torch.arange(max_len, device=sequence_output.device).unsqueeze(0)
            < text_lengths.unsqueeze(1)
        )
        mask = mask.unsqueeze(-1).float()
        masked_output = sequence_output * mask
        sum_output = masked_output.sum(dim=1)
        mean_output = sum_output / text_lengths.unsqueeze(1).float()
        pooled = torch.cat([attention_context, mean_output], dim=1)
        pooled = self.layer_norm(pooled)
        pooled = self.dropout(pooled)
        if self.use_feature_fusion and extra_features is not None:
            pooled = torch.cat([pooled, extra_features], dim=1)
        output = self.classifier(pooled)
        return output, attention_weights


In [ ]:
%writefile src/models/train_utils.py
import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from sklearn.metrics import f1_score


def train_gru_epoch(model, dataloader, optimizer, criterion, device, clip_value=1.0):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    for embeddings, pos_ids, labels, lengths in tqdm(dataloader, desc="Training"):
        embeddings = embeddings.to(device)
        labels = labels.to(device)
        lengths = lengths.to(device)
        optimizer.zero_grad()
        if getattr(model, "use_pos", False):
            predictions = model(embeddings, lengths, pos_ids=pos_ids)
        else:
            predictions = model(embeddings, lengths)
        loss = criterion(predictions, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)
        optimizer.step()
        _, predicted = torch.max(predictions, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader), correct / total


def evaluate_gru(model, dataloader, criterion, device):
    model.eval()
    epoch_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for embeddings, pos_ids, labels, lengths in tqdm(dataloader, desc="Evaluating"):
            embeddings = embeddings.to(device)
            labels = labels.to(device)
            lengths = lengths.to(device)
            if getattr(model, "use_pos", False):
                predictions = model(embeddings, lengths, pos_ids=pos_ids)
            else:
                predictions = model(embeddings, lengths)
            loss = criterion(predictions, labels)
            _, predicted = torch.max(predictions, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            epoch_loss += loss.item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return epoch_loss / len(dataloader), correct / total, all_preds, all_labels


def train_gru_epoch_with_features(
    model, dataloader, feature_tensor, optimizer, criterion, device, clip_value=1.0
):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    batch_start_idx = 0
    for embeddings, pos_ids, labels, lengths in tqdm(dataloader, desc="Training"):
        batch_size = embeddings.size(0)
        embeddings = embeddings.to(device)
        labels = labels.to(device)
        lengths = lengths.to(device)
        optimizer.zero_grad()
        extra_features = None
        if feature_tensor is not None:
            extra_features = feature_tensor[batch_start_idx : batch_start_idx + batch_size].to(
                device
            )
            batch_start_idx += batch_size
        if hasattr(model, "use_feature_fusion") and model.use_feature_fusion:
            predictions, _ = model(embeddings, lengths, extra_features, pos_ids)
        else:
            predictions = model(embeddings, lengths, pos_ids=pos_ids)
        loss = criterion(predictions, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)
        optimizer.step()
        _, predicted = torch.max(predictions, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader), correct / total


def evaluate_gru_with_features(model, dataloader, feature_tensor, criterion, device):
    model.eval()
    epoch_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    batch_start_idx = 0
    with torch.no_grad():
        for embeddings, pos_ids, labels, lengths in tqdm(dataloader, desc="Evaluating"):
            batch_size = embeddings.size(0)
            embeddings = embeddings.to(device)
            labels = labels.to(device)
            lengths = lengths.to(device)
            extra_features = None
            if feature_tensor is not None:
                extra_features = feature_tensor[batch_start_idx : batch_start_idx + batch_size].to(
                    device
                )
                batch_start_idx += batch_size
            if hasattr(model, "use_feature_fusion") and model.use_feature_fusion:
                predictions, _ = model(embeddings, lengths, extra_features, pos_ids)
            else:
                predictions = model(embeddings, lengths, pos_ids=pos_ids)
            loss = criterion(predictions, labels)
            _, predicted = torch.max(predictions, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            epoch_loss += loss.item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return epoch_loss / len(dataloader), correct / total, all_preds, all_labels


def predict_probs_with_features(model, dataloader, feature_tensor, device):
    model.eval()
    probs_list, labels_list = [], []
    batch_start_idx = 0
    with torch.no_grad():
        for embeddings, pos_ids, labels, lengths in dataloader:
            batch_size = embeddings.size(0)
            embeddings = embeddings.to(device)
            lengths = lengths.to(device)
            extra_features = None
            if feature_tensor is not None:
                extra_features = feature_tensor[batch_start_idx : batch_start_idx + batch_size].to(
                    device
                )
                batch_start_idx += batch_size
            if hasattr(model, "use_feature_fusion") and model.use_feature_fusion:
                predictions, _ = model(embeddings, lengths, extra_features, pos_ids)
            else:
                predictions = model(embeddings, lengths, pos_ids=pos_ids)
            probs_list.append(torch.softmax(predictions, dim=1).cpu().numpy())
            labels_list.append(labels.cpu().numpy())
    return np.concatenate(probs_list), np.concatenate(labels_list)


def find_best_threshold(probs, labels):
    best_threshold, best_f1 = 0.5, -1.0
    for t in np.arange(0.5, 1.0, 0.05):
        pred = (probs[:, 1] >= t).astype(int)
        f1 = f1_score(labels, pred)
        if f1 > best_f1:
            best_threshold, best_f1 = t, f1
    return float(best_threshold)


class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.0, mode="min"):
        assert mode in {"min", "max"}
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = None
        self.num_bad_epochs = 0
        self.early_stop = False
        self.best_state_dict = None
        self.best_epoch = -1
        if self.mode == "min":
            self._is_improvement = lambda current, best: (best - current) > self.min_delta
            self._init_best = float("inf")
        else:
            self._is_improvement = lambda current, best: (current - best) > self.min_delta
            self._init_best = -float("inf")
        self.best = self._init_best

    def step(self, current_value, model=None, epoch=None):
        if self.best is None:
            self.best = current_value
            if model is not None:
                self.best_state_dict = {
                    k: v.detach().clone() for k, v in model.state_dict().items()
                }
            self.best_epoch = epoch if epoch is not None else 0
            return False
        if self._is_improvement(current_value, self.best):
            self.best = current_value
            self.num_bad_epochs = 0
            self.best_epoch = epoch if epoch is not None else self.best_epoch
            if model is not None:
                self.best_state_dict = {
                    k: v.detach().clone() for k, v in model.state_dict().items()
                }
        else:
            self.num_bad_epochs += 1
            if self.num_bad_epochs >= self.patience:
                self.early_stop = True
        return self.early_stop


In [ ]:
%writefile src/models/predictor.py
import pandas as pd
import torch
import numpy as np


class SentencePredictor:
    def __init__(self, feature_pipeline_instance, trained_model, scaler, feature_columns, model_type, threshold=0.5):
        self.feature_pipeline = feature_pipeline_instance
        self.model = trained_model
        self.scaler = scaler
        self.feature_columns = feature_columns
        self.model_type = model_type
        self.threshold = threshold
        self._cache = {}
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if self.model_type == "gru":
            self.model.to(self.device)

    def clear_cache(self):
        self._cache.clear()

    def extract_features_for_sentence(self, sentence, use_cache=True):
        if use_cache and sentence in self._cache:
            return self._cache[sentence]
        temp_df = pd.DataFrame({"text": [sentence], "sentence_correct": [0]})
        processed_df = self.feature_pipeline.extract_all_features(temp_df)
        if use_cache:
            self._cache[sentence] = processed_df
        return processed_df

    def predict_sentence(self, sentence, use_cache=True):
        processed_df = self.extract_features_for_sentence(sentence, use_cache=use_cache)
        features_dict = processed_df[self.feature_columns].iloc[0].to_dict()
        tokens = processed_df["tokens"].iloc[0]
        pos_tags = processed_df["pos_tags"].iloc[0]
        prediction_numeric = None
        confidence = None
        prediction_label = None
        if self.model_type == "ml":
            feature_vector = processed_df[self.feature_columns].iloc[0].values.reshape(1, -1)
            feature_vector = self.scaler.transform(feature_vector)
            if hasattr(self.model, "predict_proba"):
                proba = self.model.predict_proba(feature_vector)[0]
                prediction_numeric = self.model.predict(feature_vector)[0]
                confidence = max(proba)
            else:
                prediction_numeric = self.model.predict(feature_vector)[0]
                confidence = None
        elif self.model_type == "gru":
            feature_vector = processed_df[self.feature_columns].iloc[0].values.reshape(1, -1)
            feature_vector_scaled = self.scaler.transform(feature_vector)
            extra_features = torch.FloatTensor(feature_vector_scaled).to(self.device)
            temp_dataset = self.feature_pipeline.create_temp_dataset(tokens, pos_tags)
            self.model.eval()
            with torch.no_grad():
                embeddings, pos_ids, _, lengths = temp_dataset[0]
                embeddings = embeddings.unsqueeze(0).to(self.device)
                pos_ids = pos_ids.unsqueeze(0).to(self.device)
                lengths = torch.LongTensor([lengths]).to(self.device)
                if hasattr(self.model, "use_feature_fusion") and self.model.use_feature_fusion:
                    output, _ = self.model(embeddings, lengths, extra_features, pos_ids)
                else:
                    output = self.model(embeddings, lengths, pos_ids=pos_ids)
                probs = torch.softmax(output, dim=1)
                prediction_numeric = 1 if probs[0, 1].item() >= self.threshold else 0
                confidence = probs[0, prediction_numeric].item()
        prediction_label = "Right" if prediction_numeric == 1 else "Wrong"
        return {
            "sentence": sentence,
            "prediction": prediction_label,
            "prediction_numeric": prediction_numeric,
            "confidence": confidence,
            "features": features_dict,
            "tokens": tokens,
            "pos_tags": pos_tags,
        }

    def predict_batch(self, sentences, use_cache=True):
        results = []
        for sentence in sentences:
            result = self.predict_sentence(sentence, use_cache=use_cache)
            results.append(result)
        return results


In [ ]:
%writefile src/pipeline/__init__.py
from .feature_pipeline import FeaturePipeline

__all__ = ["FeaturePipeline"]


In [ ]:
%writefile src/pipeline/feature_pipeline.py
import pandas as pd
from src.utils import load_fasttext_model
from src.data.pos_tagger import KhmerPOSTagger
from src.features.oov import EmbeddingOOVCalculator
from src.features.grammar import SimplePOSGrammarExtractor
from src.features.structural import StructuralFeatureExtractor


class FeaturePipeline:
    def __init__(self, embedding_model_path=None, embedding_model=None, lazy_pos=True):
        if embedding_model is not None:
            self.embedding_model = embedding_model
        elif embedding_model_path:
            self.embedding_model = load_fasttext_model(embedding_model_path)
        else:
            raise ValueError("Provide embedding_model or embedding_model_path")
        self.lazy_pos = lazy_pos
        self.pos_tagger = None if lazy_pos else KhmerPOSTagger()
        self.oov_extractor = EmbeddingOOVCalculator(self.embedding_model)
        self.grammar_extractor = SimplePOSGrammarExtractor()
        self.structural_extractor = StructuralFeatureExtractor()

    def _ensure_pos_tagger(self):
        if self.pos_tagger is None:
            self.pos_tagger = KhmerPOSTagger()

    def extract_all_features(self, df):
        if "text" not in df.columns:
            raise ValueError("DataFrame must contain a 'text' column for POS tagging.")
        self._ensure_pos_tagger()
        df = self.pos_tagger.tag_dataframe(df)
        df = self.oov_extractor.add_oov_features(df)
        df = self.grammar_extractor.extract_features(df)
        df = self.structural_extractor.extract_features(df)
        return df

    def create_temp_dataset(self, tokens, pos_tags=None):
        from src.data.dataset import KhmerTextDataset

        return KhmerTextDataset(
            [tokens], pos_tags if pos_tags is not None else [[]], [0], self.embedding_model, use_cache=True
        )

    @staticmethod
    def get_feature_cols():
        return [
            "oov_ratio",
            "dep_grammar_score",
            "sentence_length",
            "pos_diversity",
            "avg_word_length",
        ]


In [ ]:
%writefile train.py
import gc
import torch
import torch.nn as nn
import torch.optim as optim

from src.config import (
    EMBEDDING_DIM,
    HIDDEN_DIM,
    OUTPUT_DIM,
    N_LAYERS,
    DROPOUT,
    LEARNING_RATE,
    N_EPOCHS,
    BATCH_SIZE,
    PATIENCE,
    CLIP_VALUE,
    DEVICE,
    DATA_PATH,
    MODEL_SAVE_PATH,
    FASTTEXT_MODEL_PATH,
    FEATURE_COLUMNS,
    USE_FEATURE_FUSION,
    NUM_EXTRA_FEATURES,
    CLASS_WEIGHTS,
    POS_TAGS,
    POS_EMBEDDING_DIM,
)
from src.utils import set_seed, load_and_split_data, create_dataloaders, build_feature_tensor, load_fasttext_model
from src.data.dataset import KhmerTextDataset
from src.models.gru import ImprovedGRUClassifier
from src.models.train_utils import (
    train_gru_epoch_with_features,
    evaluate_gru_with_features,
    predict_probs_with_features,
    find_best_threshold,
    EarlyStopping,
)


def main():
    set_seed()
    print(f"Using device: {DEVICE}")

    print("Loading FastText embedding model...")
    embedding_model = load_fasttext_model(FASTTEXT_MODEL_PATH)

    print("\nLoading and splitting data...")
    df, X_train, X_val, X_test, y_train, y_val, y_test, scaler, X_train_scaled, X_val_scaled, X_test_scaled = load_and_split_data(
        DATA_PATH, FEATURE_COLUMNS
    )

    print("\nCreating datasets...")
    train_dataset = KhmerTextDataset(
        df.loc[X_train.index, "tokens"].tolist() if "tokens" in df.columns else df.loc[X_train.index, "text"].tolist(),
        df.loc[X_train.index, "pos_tags"].tolist() if "pos_tags" in df.columns else [[] for _ in range(len(X_train))],
        y_train.values,
        embedding_model,
    )
    val_dataset = KhmerTextDataset(
        df.loc[X_val.index, "tokens"].tolist() if "tokens" in df.columns else df.loc[X_val.index, "text"].tolist(),
        df.loc[X_val.index, "pos_tags"].tolist() if "pos_tags" in df.columns else [[] for _ in range(len(X_val))],
        y_val.values,
        embedding_model,
    )
    test_dataset = KhmerTextDataset(
        df.loc[X_test.index, "tokens"].tolist() if "tokens" in df.columns else df.loc[X_test.index, "text"].tolist(),
        df.loc[X_test.index, "pos_tags"].tolist() if "pos_tags" in df.columns else [[] for _ in range(len(X_test))],
        y_test.values,
        embedding_model,
    )

    print(f"Train Dataset size: {len(train_dataset)}")
    print(f"Validation Dataset size: {len(val_dataset)}")
    print(f"Test Dataset size: {len(test_dataset)}")

    train_loader, val_loader, test_loader = create_dataloaders(
        train_dataset, val_dataset, test_dataset, BATCH_SIZE
    )

    print("\nInitializing Improved GRU model with feature fusion...")
    gru_model = ImprovedGRUClassifier(
        EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT,
        num_extra_features=NUM_EXTRA_FEATURES, use_feature_fusion=USE_FEATURE_FUSION,
        pos_vocab_size=len(POS_TAGS), pos_embedding_dim=POS_EMBEDDING_DIM,
    ).to(DEVICE)
    optimizer = optim.Adam(gru_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss(
        weight=torch.FloatTensor(CLASS_WEIGHTS).to(DEVICE)
    )

    print("=" * 80)
    print("Training Improved GRU Model (with feature fusion)")
    print("=" * 80)
    print(f"Embedding Dim: {EMBEDDING_DIM}")
    print(f"Hidden Dim: {HIDDEN_DIM}")
    print(f"Num Layers: {N_LAYERS}")
    print(f"Dropout: {DROPOUT}")
    print(f"Feature Fusion: {USE_FEATURE_FUSION}")
    print(f"Num Extra Features: {NUM_EXTRA_FEATURES}")
    print(f"Learning Rate: {LEARNING_RATE}")
    print(f"Batch Size: {BATCH_SIZE}")
    print(f"Epochs: {N_EPOCHS}")
    print(f"Early Stopping Patience: {PATIENCE}")
    print("=" * 80)

    train_feat_tensor = torch.FloatTensor(X_train_scaled)
    val_feat_tensor = torch.FloatTensor(X_val_scaled)

    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    best_val_acc = 0.0
    best_model_state = None
    early_stopper = EarlyStopping(patience=PATIENCE, min_delta=0.0, mode="min")

    for epoch in range(N_EPOCHS):
        print(f"\nEpoch {epoch + 1}/{N_EPOCHS}")
        print("-" * 80)
        train_loss, train_acc = train_gru_epoch_with_features(
            gru_model, train_loader, train_feat_tensor, optimizer, criterion, DEVICE, CLIP_VALUE
        )
        val_loss, val_acc, _, _ = evaluate_gru_with_features(
            gru_model, val_loader, val_feat_tensor, criterion, DEVICE
        )
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc * 100:.2f}%")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc * 100:.2f}%")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        early_stopper.step(val_loss, gru_model, epoch=epoch + 1)
        if early_stopper.early_stop:
            print(
                f"Early stopping triggered at epoch {epoch + 1}. "
                f"Restoring best weights from epoch {early_stopper.best_epoch}."
            )
            break

    print("\n" + "=" * 80)
    print("Training Complete!")
    print(f"Best Validation Accuracy (observed): {best_val_acc * 100:.2f}%")
    if early_stopper.best_epoch != -1:
        print(
            f"Best model (by val_loss) was at epoch {early_stopper.best_epoch} "
            f"with value {early_stopper.best:.4f}"
        )
    print("=" * 80)

    if early_stopper.best_state_dict is not None:
        best_model_state = early_stopper.best_state_dict
    if best_model_state is not None:
        gru_model.load_state_dict(best_model_state)
        val_probs, _ = predict_probs_with_features(gru_model, val_loader, val_feat_tensor, DEVICE)
        threshold = find_best_threshold(val_probs, y_val.values)
        torch.save(
            {
                "model_state_dict": best_model_state,
                "scaler": scaler,
                "feature_columns": FEATURE_COLUMNS,
                "threshold": threshold,
            },
            MODEL_SAVE_PATH,
        )
        print(f"Best decision threshold (P(Right) >= {threshold:.3f}) saved to {MODEL_SAVE_PATH}")
    else:
        print("Warning: No best model state captured; using final epoch weights.")
        torch.save(
            {
                "model_state_dict": gru_model.state_dict(),
                "scaler": scaler,
                "feature_columns": FEATURE_COLUMNS,
                "threshold": 0.5,
            },
            MODEL_SAVE_PATH,
        )


if __name__ == "__main__":
    main()


In [ ]:
%writefile evaluate.py
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

from src.config import (
    EMBEDDING_DIM,
    HIDDEN_DIM,
    OUTPUT_DIM,
    N_LAYERS,
    DROPOUT,
    BATCH_SIZE,
    DEVICE,
    DATA_PATH,
    MODEL_SAVE_PATH,
    FEATURE_COLUMNS,
    USE_FEATURE_FUSION,
    NUM_EXTRA_FEATURES,
    POS_TAGS,
    POS_EMBEDDING_DIM,
)
from src.utils import load_and_split_data, create_dataloaders, build_feature_tensor
from src.data.dataset import KhmerTextDataset
from src.models.gru import ImprovedGRUClassifier
from src.models.train_utils import predict_probs_with_features


def main():
    print(f"Using device: {DEVICE}")

    print("Loading FastText embedding model...")
    embedding_model = fasttext.load_model("cc.km.300.bin")

    print("\nLoading and splitting data...")
    df, X_train, X_val, X_test, y_train, y_val, y_test, scaler, _, _, _ = load_and_split_data(
        DATA_PATH, FEATURE_COLUMNS
    )

    print("\nCreating test dataset...")
    test_dataset = KhmerTextDataset(
        df.loc[X_test.index, "tokens"].tolist() if "tokens" in df.columns else df.loc[X_test.index, "text"].tolist(),
        df.loc[X_test.index, "pos_tags"].tolist() if "pos_tags" in df.columns else [[] for _ in range(len(X_test))],
        y_test.values,
        embedding_model,
    )
    _, _, test_loader = create_dataloaders(None, None, test_dataset, BATCH_SIZE)

    print("\nLoading saved model...")
    checkpoint = torch.load(MODEL_SAVE_PATH, map_location=DEVICE, weights_only=False)
    model_state = checkpoint["model_state_dict"]
    scaler = checkpoint.get("scaler", None)
    threshold = checkpoint.get("threshold", 0.5)

    gru_model = ImprovedGRUClassifier(
        EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT,
        num_extra_features=NUM_EXTRA_FEATURES, use_feature_fusion=USE_FEATURE_FUSION,
        pos_vocab_size=len(POS_TAGS), pos_embedding_dim=POS_EMBEDDING_DIM,
    ).to(DEVICE)
    gru_model.load_state_dict(model_state)

    test_feat_tensor = build_feature_tensor(df, X_test.index, FEATURE_COLUMNS, scaler)

    print("=" * 80)
    print("Evaluating Improved GRU on Test Set (with feature fusion)")
    print(f"Decision threshold: P(Right) >= {threshold:.3f}")
    print("=" * 80)

    test_probs, y_true = predict_probs_with_features(
        gru_model, test_loader, test_feat_tensor, DEVICE
    )
    y_pred = (test_probs[:, 1] >= threshold).astype(int)

    test_acc = accuracy_score(y_true, y_pred)
    test_precision = precision_score(y_true, y_pred)
    test_recall = recall_score(y_true, y_pred)
    test_f1 = f1_score(y_true, y_pred)

    print(f"\nGRU Test Results:")
    print(f"  Accuracy:  {test_acc:.4f}")
    print(f"  Precision: {test_precision:.4f}")
    print(f"  Recall:    {test_recall:.4f}")
    print(f"  F1-Score:  {test_f1:.4f}")

    print("\nDetailed Classification Report:")
    print(
        classification_report(
            y_true, y_pred, target_names=["Wrong", "Right"]
        )
    )

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Greens",
        xticklabels=["Wrong", "Right"],
        yticklabels=["Wrong", "Right"],
    )
    plt.title("Confusion Matrix - GRU")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.savefig("gru_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()


In [ ]:
%writefile predict.py
import sys
import torch
import fasttext
import pandas as pd

from src.config import (
    EMBEDDING_DIM,
    HIDDEN_DIM,
    OUTPUT_DIM,
    N_LAYERS,
    DROPOUT,
    DEVICE,
    FASTTEXT_MODEL_PATH,
    MODEL_SAVE_PATH,
    USE_FEATURE_FUSION,
    NUM_EXTRA_FEATURES,
    POS_TAGS,
    POS_EMBEDDING_DIM,
)
from src.models.gru import ImprovedGRUClassifier
from src.pipeline.feature_pipeline import FeaturePipeline
from src.models.predictor import SentencePredictor


def main():
    if len(sys.argv) < 2:
        print("Usage: python predict.py <sentence>")
        sys.exit(1)

    sentence = sys.argv[1]

    print("Loading FastText model...")
    embedding_model = fasttext.load_model(FASTTEXT_MODEL_PATH)

    print("Loading saved model checkpoint...")
    checkpoint = torch.load(MODEL_SAVE_PATH, map_location=DEVICE, weights_only=False)

    if "feature_columns" in checkpoint:
        feature_columns = checkpoint["feature_columns"]
    else:
        from src.config import FEATURE_COLUMNS
        feature_columns = FEATURE_COLUMNS

    scaler = checkpoint.get("scaler", None)
    if scaler is None:
        print("Warning: No scaler found in checkpoint.")
    threshold = checkpoint.get("threshold", 0.5)

    gru_model = ImprovedGRUClassifier(
        EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT,
        num_extra_features=NUM_EXTRA_FEATURES, use_feature_fusion=USE_FEATURE_FUSION,
        pos_vocab_size=len(POS_TAGS), pos_embedding_dim=POS_EMBEDDING_DIM,
    ).to(DEVICE)
    gru_model.load_state_dict(checkpoint["model_state_dict"])
    gru_model.eval()

    print("Initializing feature pipeline...")
    feature_pipeline = FeaturePipeline(embedding_model=embedding_model, lazy_pos=True)

    print("Creating predictor...")
    predictor = SentencePredictor(
        feature_pipeline_instance=feature_pipeline,
        trained_model=gru_model,
        scaler=scaler,
        feature_columns=feature_columns,
        model_type="gru",
        threshold=threshold,
    )

    print(f"\nPredicting for: {sentence}")
    result = predictor.predict_sentence(sentence)

    print(f"Prediction: {result['prediction']} (numeric: {result['prediction_numeric']})")
    if result["confidence"] is not None:
        print(f"Confidence: {result['confidence']:.4f}")

    print("\nExtracted Features:")
    for k, v in result["features"].items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    print(f"\nTokens: {result['tokens']}")
    print(f"POS Tags: {result['pos_tags']}")


if __name__ == "__main__":
    main()


In [ ]:
%writefile train_transformer.py
"""
Fine-tune a pretrained Khmer transformer (seanghay/xlm-roberta-khmer-small)
for the binary sentence-grammar task. Replaces the FastText + BiGRU + manual
feature pipeline entirely: the transformer reads raw Khmer text directly.

Usage:
    python train_transformer.py
    python evaluate_transformer.py
"""

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

from src.config import DATA_PATH, MODEL_SAVE_PATH, CLASS_WEIGHTS, SEED

MODEL_NAME = "seanghay/xlm-roberta-khmer-small"
MAX_LENGTH = 128
BATCH_SIZE = 16
N_EPOCHS = 3
LEARNING_RATE = 2e-5
TRANSFORMER_SAVE_PATH = "transformer_model.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_splits():
    df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
    df = df[["text", "sentence_correct"]]
    X_temp, X_test, y_temp, y_test = train_test_split(
        df["text"], df["sentence_correct"], test_size=0.2, random_state=SEED, stratify=df["sentence_correct"]
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=SEED, stratify=y_temp
    )
    print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")
    return X_train, X_val, X_test, y_train, y_val, y_test


def make_batches(texts, labels, tokenizer):
    for i in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts.iloc[i : i + BATCH_SIZE].tolist()
        batch_labels = torch.LongTensor(labels.iloc[i : i + BATCH_SIZE].values)
        enc = tokenizer(
            batch_texts, padding="longest", truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        )
        yield enc, batch_labels


def run_epoch(model, batches, optimizer, criterion, device, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for enc, labels in batches:
        enc = {k: v.to(device) for k, v in enc.items()}
        labels = labels.to(device)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits = model(**enc).logits
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
        probs = torch.softmax(logits, dim=1)
        all_probs.append(probs[:, 1].detach().cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        total_loss += loss.item() * labels.size(0)
        correct += (probs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return (
        total_loss / total,
        correct / total,
        np.concatenate(all_probs),
        np.concatenate(all_labels),
    )


def find_best_threshold(probs, labels):
    best_t, best_f1 = 0.5, -1.0
    for t in np.arange(0.5, 1.0, 0.05):
        f1 = f1_score(labels, probs >= t)
        if f1 > best_f1:
            best_t, best_f1 = t, f1
    return float(best_t)


def main():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    print(f"Using device: {DEVICE}")
    X_train, X_val, X_test, y_train, y_val, y_test = load_splits()

    print(f"Loading tokenizer + model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(CLASS_WEIGHTS).to(DEVICE))

    best_val_f1, best_state, best_threshold = 0.0, None, 0.5
    for epoch in range(N_EPOCHS):
        train_loss, train_acc, _, _ = run_epoch(
            model, make_batches(X_train, y_train, tokenizer), optimizer, criterion, DEVICE, train=True
        )
        val_loss, val_acc, val_probs, val_labels = run_epoch(
            model, make_batches(X_val, y_val, tokenizer), optimizer, criterion, DEVICE, train=False
        )
        val_f1 = f1_score(val_labels, val_probs >= 0.5)
        print(
            f"Epoch {epoch + 1}/{N_EPOCHS} | Train loss {train_loss:.4f} acc {train_acc:.4f} "
            f"| Val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f}"
        )
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            best_threshold = find_best_threshold(val_probs, val_labels)

    model.load_state_dict(best_state)
    torch.save(
        {
            "model_state_dict": best_state,
            "tokenizer_name": MODEL_NAME,
            "threshold": best_threshold,
            "num_labels": 2,
        },
        TRANSFORMER_SAVE_PATH,
    )
    print(f"Best val F1 {best_val_f1:.4f} at threshold {best_threshold:.3f}. Saved to {TRANSFORMER_SAVE_PATH}")

    print("\nEvaluating on test set...")
    _, _, test_probs, test_labels = run_epoch(
        model, make_batches(X_test, y_test, tokenizer), optimizer, criterion, DEVICE, train=False
    )
    y_pred = (test_probs >= best_threshold).astype(int)
    print(f"Accuracy:  {accuracy_score(test_labels, y_pred):.4f}")
    print(f"Precision: {precision_score(test_labels, y_pred):.4f}")
    print(f"Recall:    {recall_score(test_labels, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(test_labels, y_pred):.4f}")
    print("\n" + classification_report(test_labels, y_pred, target_names=["Wrong", "Right"]))


if __name__ == "__main__":
    main()


In [ ]:
%writefile scripts/generate_errors.py
"""
Generate a grammar-checking dataset from a raw Khmer sentence corpus.

Produces a balanced labeled CSV where:
  - label 1 = the original sentence (correct)
  - label 0 = a corrupted version, one of several error types

CRITICAL: corrupted sentences are written as the token list JOINED WITH NO
SPACES ("".join(tokens)), matching how Khmer is actually written. The previous
dataset wrote scrambled tokens space-separated, which made "contains a space"
a near-perfect Wrong detector (100% of Wrong rows had a space vs 49.9% of
Right rows) -- the classifier was memorizing spacing, not grammar.

Also propagates an optional `doc_id` column so you can split train/test by
source document and avoid leakage between a sentence and its corrupted twin.

Usage:
    # plain Khmer sentences, one per line; tagged with khmer-pos-roberta:
    python scripts/generate_errors.py corpus.txt -o data_augmented.csv
    # already-tagged CSV (columns: text,tokens,pos_tags[,doc_id]):
    python scripts/generate_errors.py tagged.csv --pretagged -o data_augmented.csv
"""

import argparse
import ast
import json
import random
from pathlib import Path

import pandas as pd

PARTS = [
    "គឺ", "ជា", "ក៏", "ដើម្បី", "និង", "ដែល", "បាន", "ទៅ", "មក", "នៅ",
]

SUBJECT_TAGS = ("NN", "PN", "PRO", "PR")
VERB_TAGS = ("VB", "VB_JJ", "AUX")


def error_shuffle(tokens):
    idx = list(range(len(tokens)))
    random.shuffle(idx)
    return [tokens[i] for i in idx]


def error_swap_adjacent(tokens):
    if len(tokens) < 2:
        return tokens
    i = random.randrange(len(tokens) - 1)
    out = list(tokens)
    out[i], out[i + 1] = out[i + 1], out[i]
    return out


def error_drop_token(tokens):
    if len(tokens) <= 1:
        return tokens
    i = random.randrange(len(tokens))
    return [t for j, t in enumerate(tokens) if j != i]


def error_drop_subject(tokens, pos_tags):
    for i, tag in enumerate(pos_tags):
        if tag.startswith(SUBJECT_TAGS):
            return [t for j, t in enumerate(tokens) if j != i]
    return tokens


def error_front_verb(tokens, pos_tags):
    verb_i = next((i for i, t in enumerate(pos_tags) if t.startswith(VERB_TAGS)), None)
    if verb_i is None:
        return tokens
    out = list(tokens)
    out.insert(0, out.pop(verb_i))
    return out


def error_drop_particle(tokens, pos_tags):
    for i, tok in enumerate(tokens):
        if tok in PARTS:
            return [t for j, t in enumerate(tokens) if j != i]
    return tokens


def error_duplicate(tokens):
    if len(tokens) < 2:
        return tokens
    i = random.randrange(len(tokens))
    out = list(tokens)
    out.insert(i, out[i])
    return out


def error_fragment(tokens):
    if len(tokens) <= 2:
        return tokens
    return tokens[: random.randint(2, 3)]


ERROR_TYPES = [
    ("shuffle", error_shuffle),
    ("swap_adjacent", error_swap_adjacent),
    ("drop_token", error_drop_token),
    ("drop_subject", error_drop_subject),
    ("front_verb", error_front_verb),
    ("drop_particle", error_drop_particle),
    ("duplicate", error_duplicate),
    ("fragment", error_fragment),
]


def tag_corpus(texts):
    from src.data.pos_tagger import KhmerPOSTagger

    tagger = KhmerPOSTagger()
    tokens, pos_tags = [], []
    for text in texts:
        tok, pos = tagger.tag_sentence(text)
        tokens.append(tok)
        pos_tags.append(pos)
    return tokens, pos_tags


def make_row(text, tokens, pos_tags, label, error_type, doc_id):
    # join without spaces so spacing is never a predictive feature
    return {
        "text": "".join(tokens) if label == 0 else text,
        "tokens": tokens,
        "pos_tags": pos_tags,
        "sentence_correct": label,
        "error_type": error_type,
        "doc_id": doc_id,
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("input", help="Path to corpus (text file or CSV).")
    ap.add_argument("--pretagged", action="store_true", help="CSV already has tokens+pos_tags columns.")
    ap.add_argument("-o", "--output", default="data_augmented.csv")
    ap.add_argument("--negatives-per-positive", type=int, default=4)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--doc-id-col", default="doc_id")
    args = ap.parse_args()
    random.seed(args.seed)

    path = Path(args.input)
    if path.suffix.lower() in (".csv", ".tsv") or args.pretagged:
        df = pd.read_csv(path, encoding="utf-8-sig")
        if "tokens" in df.columns:
            df["tokens"] = df["tokens"].apply(ast.literal_eval)
        if "pos_tags" in df.columns:
            df["pos_tags"] = df["pos_tags"].apply(ast.literal_eval)
        texts = df["text"].tolist()
        tokens = df["tokens"].tolist() if "tokens" in df.columns else None
        pos_tags = df["pos_tags"].tolist() if "pos_tags" in df.columns else None
        doc_ids = df[args.doc_id_col].tolist() if args.doc_id_col in df.columns else list(range(len(df)))
        if tokens is None or pos_tags is None:
            tokens, pos_tags = tag_corpus(texts)
    else:
        with open(path, encoding="utf-8") as f:
            texts = [ln.strip() for ln in f if ln.strip()]
        tokens, pos_tags = tag_corpus(texts)
        doc_ids = list(range(len(texts)))

    rows = []
    for text, tok, pos, doc_id in zip(texts, tokens, pos_tags, doc_ids):
        if len(tok) < 3:
            continue
        rows.append(make_row(text, tok, pos, 1, "original", doc_id))
        candidates = list(ERROR_TYPES)
        random.shuffle(candidates)
        made = 0
        for name, fn in candidates:
            if made >= args.negatives_per_positive:
                break
            corrupt = fn(tok, pos) if name in ("drop_subject", "front_verb", "drop_particle") else fn(tok)
            if corrupt == tok or len(corrupt) == 0:
                continue
            rows.append(make_row(text, corrupt, pos, 0, name, doc_id))
            made += 1

    out = pd.DataFrame(rows)
    out.to_csv(args.output, index=False, encoding="utf-8-sig")
    print(f"Wrote {len(out):,} rows to {args.output}")
    print(out["sentence_correct"].value_counts().to_dict())
    print("\nError-type distribution:")
    print(out["error_type"].value_counts())


if __name__ == "__main__":
    main()


## 3. Data quality check — the "space" leak

Verify the artifact that motivated the data overhaul. A healthy dataset should
show roughly the same spacing pattern for both classes.


In [ ]:
import csv

rows = list(csv.reader(open('train_data.csv', encoding='utf-8-sig')))[1:]

def has_space(t):
    return ' ' in t

sp = {0: 0, 1: 0}
tot = {0: 0, 1: 0}
for r in rows:
    lab = int(r[3])
    tot[lab] += 1
    sp[lab] += has_space(r[0])

print('RIGHT: %d rows, %d contain a space (%.1f%%)' % (tot[1], sp[1], 100 * sp[1] / tot[1]))
print('WRONG: %d rows, %d contain a space (%.1f%%)' % (tot[0], sp[0], 100 * sp[0] / tot[0]))


## 4. Train the BiGRU

Splits data, trains with feature fusion + POS embeddings + class weights,
tunes the decision threshold on validation, saves `gru_model.pth`.


In [ ]:
!python train.py

## 5. Evaluate

Loads the checkpoint, applies the saved threshold, prints a correct
`Wrong/Right` report + confusion matrix.


In [ ]:
!python evaluate.py

## 6. Try it on single sentences

One clearly correct sentence and one scrambled (incorrect) sentence.


In [ ]:
# A clearly correct sentence, then a scrambled (incorrect) one:
!python predict.py "កុមារកម្ពុជាទៅសាលារៀននៅព្រឹកនេះ"
!python predict.py "បាន ស៊ុត នាទីទី ដំបូង ពាក់កណ្តាល សំរាប់ អ៊ីតាលី"


## 7. (Optional) Fine-tune the Khmer transformer

Highest expected accuracy ceiling. Trains on raw text — no FastText or
feature pipeline. Saves `transformer_model.pth`. Needs the `transformers`
model download (~200MB); skip this cell to keep runtime short.


In [ ]:
# Optional: downloads the model + trains 3 epochs. Comment out to skip.
!python train_transformer.py


## 8. (Optional) Generate a new dataset with the error taxonomy

Feed a raw Khmer corpus (one sentence per line). Corruptions are written
space-free so spacing can never become a predictive feature.


In [ ]:
!python scripts/generate_errors.py --help